# Lab 1 — Trace an Internal-Operations Assistant

**Required · 45 minutes**

Instrument one synthetic incident-assistance request with OpenTelemetry, export it to the project-connected Application Insights resource, and record privacy-safe operational attributes. Client-side GenAI instrumentation is currently preview; custom OpenTelemetry spans provide a stable fallback.

**Artifact:** a trace ID plus a screenshot or URL from Foundry/Application Insights.

## 1. Configuration and namespace

Authenticate with `az login` before running this lab. `DefaultAzureCredential` also supports other configured developer or managed-identity credentials.

Content capture is deliberately off. Enabling it records prompts, model outputs, and tool parameters; use only synthetic data in an approved development environment.

In [ ]:
import os
import re
from pathlib import Path
from importlib.metadata import version
from dotenv import load_dotenv

def load_repo_env() -> Path | None:
    start = Path.cwd().resolve()
    for folder in (start, *start.parents):
        candidate = folder / '.env'
        if candidate.exists():
            load_dotenv(candidate)
            return candidate
    return None

env_path = load_repo_env()
endpoint = os.getenv('FOUNDRY_PROJECT_ENDPOINT') or os.getenv('AZURE_AI_PROJECT_ENDPOINT')
model_deployment = os.getenv('FOUNDRY_MODEL') or os.getenv('AZURE_AI_MODEL_DEPLOYMENT_NAME')
team_id = os.getenv('WORKSHOP_TEAM_ID', '').strip()
participant_id = os.getenv('WORKSHOP_PARTICIPANT_ID', '').strip()
configured_namespace = os.getenv('WORKSHOP_RESOURCE_NAMESPACE', '').strip()
raw_namespace = configured_namespace or team_id or participant_id
resource_namespace = re.sub(r'[^a-z0-9-]+', '-', raw_namespace.lower()).strip('-')[:32]
if not endpoint or not model_deployment or not resource_namespace:
    raise ValueError('Set the Foundry endpoint, model deployment, and workshop namespace')

def workshop_name(prefix: str) -> str:
    return f'{prefix}-{resource_namespace}'

print({'env': str(env_path) if env_path else 'process environment', 'namespace': resource_namespace, 'team': team_id, 'participant': participant_id})
print('azure-ai-projects', version('azure-ai-projects'))

## 2. Enable and export tracing

The opt-in variables must be set before instrumentation. Trace-context propagation is enabled so client and server spans can correlate. Message content stays disabled.

In [ ]:
os.environ['AZURE_EXPERIMENTAL_ENABLE_GENAI_TRACING'] = 'true'
os.environ['OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT'] = 'false'
os.environ['AZURE_TRACING_GEN_AI_ENABLE_TRACE_CONTEXT_PROPAGATION'] = 'true'

from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.telemetry import AIProjectInstrumentor
from azure.monitor.opentelemetry import configure_azure_monitor
from opentelemetry import trace

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=endpoint, credential=credential)
connection_string = project_client.telemetry.get_application_insights_connection_string()
if not connection_string:
    raise RuntimeError('Connect Application Insights to the Foundry project before this lab')

configure_azure_monitor(connection_string=connection_string)
AIProjectInstrumentor().instrument(
    enable_content_recording=False,
    enable_trace_context_propagation=True,
    enable_baggage_propagation=False,
)
tracer = trace.get_tracer('workshop.internal_operations')
# Create the OpenAI client only after instrumentation so trace context propagates.
openai_client = project_client.get_openai_client()
print('Tracing configured; sensitive message capture is disabled.')

## 3. Create a namespaced prompt agent

The agent uses only synthetic procedure facts embedded in its instructions. Retrieval with Foundry IQ or Azure AI Search belongs to the knowledge/retrieval layer; tracing observes either implementation.

In [ ]:
from azure.ai.projects.models import PromptAgentDefinition

agent_name = workshop_name('d2-observe')
agent = project_client.agents.create_version(
    agent_name=agent_name,
    definition=PromptAgentDefinition(
        model=model_deployment,
        instructions=(
            'You are a synthetic internal-operations assistant. '
            'Use only these workshop facts: P-17 requires isolation, absence-of-voltage verification, '
            'and a switching-authority confirmation before dispatch. '
            'Never claim that an operational action was executed. Cite procedure P-17. '
            'If required information is absent, state what must be verified.'
        ),
    ),
)
conversation = openai_client.conversations.create()
print({'agent': agent.name, 'version': agent.version, 'conversation': conversation.id})

## 4. Run one traced scenario

Custom attributes must be low-cardinality and privacy-safe. Do not add names, raw prompts, secrets, access tokens, or production incident descriptions.

In [ ]:
synthetic_incident_id = 'SIM-1042'
with tracer.start_as_current_span('workshop.incident_assistance') as span:
    span.set_attribute('workshop.namespace', resource_namespace)
    span.set_attribute('workshop.team_id', team_id)
    span.set_attribute('workshop.scenario', 'procedure-guidance')
    span.set_attribute('incident.synthetic_id', synthetic_incident_id)
    response = openai_client.responses.create(
        conversation=conversation.id,
        extra_body={
            'agent_reference': {
                'name': agent.name,
                'id': agent.id,
                'type': 'agent_reference',
            }
        },
        input='For synthetic incident SIM-1042, what must be verified before dispatch?',
    )
    span.set_attribute('workshop.outcome', 'response-created')
    trace_id = f'{span.get_span_context().trace_id:032x}'

print(response.output_text)
print('Trace ID:', trace_id)

## 5. Success check

In [ ]:
answer = response.output_text.lower()
assert len(trace_id) == 32 and int(trace_id, 16) > 0
assert 'p-17' in answer, 'The synthetic answer must cite P-17'
assert any(term in answer for term in ('verify', 'verification', 'confirmation'))
print('PASS — response contract and local trace ID checks succeeded.')

## Participant challenge

Add a custom child span named `workshop.procedure_lookup` around a synthetic dictionary lookup. Record only `procedure.id`, `lookup.hit`, and `workshop.namespace`. Do **not** record the query text or full procedure body.

Then verify that the child span shares the same trace ID as its parent.

In [ ]:
# TODO: implement the privacy-safe child span.
PROCEDURES = {'P-17': {'requires_switching_authority': True}}

# with tracer.start_as_current_span('workshop.procedure_lookup') as lookup_span:
#     ...


## Inspect the trace

Open the project **Tracing** view or Application Insights transaction search and filter by the trace ID. Ingestion normally takes 2–5 minutes.

Confirm that the agent/model call is correlated with `workshop.incident_assistance`, custom attributes are present, message bodies are absent, and latency/error status are visible.

## Optional extension

Add a `SpanProcessor` that injects the workshop namespace into every span, or evaluate recent Application Insights traces with Foundry trace evaluation. Trace evaluation is preview; the deterministic inline datasets in the next labs are the fallback.

## Cleanup (opt-in and namespace-safe)

Resources are retained by default so the trace remains inspectable. The cell refuses to delete an agent outside this notebook's namespace.

In [ ]:
allow_cleanup = os.getenv('WORKSHOP_ALLOW_CLEANUP', 'false').lower() == 'true'
expected_suffix = f'-{resource_namespace}'
if allow_cleanup:
    if not agent.name.endswith(expected_suffix):
        raise RuntimeError(f'Refusing to delete non-owned agent: {agent.name}')
    openai_client.conversations.delete(conversation_id=conversation.id)
    project_client.agents.delete_version(agent_name=agent.name, agent_version=agent.version)
    print('Deleted this lab conversation and namespaced agent version.')
else:
    print('Cleanup disabled. Set WORKSHOP_ALLOW_CLEANUP=true to remove only this namespaced agent.')